# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Binary classification.**

The question is: *"Will this content page decline in the next measurement window?"* — a yes/no outcome for each page. That maps directly to binary classification.

Why not the others?
- **Ranking/scoring** would answer *"which pages first?"* — useful, but the natural first step is a classifier whose predicted probabilities *produce* a ranking for free. Classification comes first; ranking is a downstream use of the output.
- **Clustering** would answer *"what kinds of pages exist?"* — interesting for exploration but doesn't directly tell an editor which page needs work.
- **Signal analysis** would answer *"which features travel together?"* — that's EDA, not the end product.

A binary classifier predicting `is_declining` (1 = page declined, 0 = did not) gives the clearest, most actionable output: a probability per page, which editors use to prioritize refreshes.

In [1]:
# Confirm the task type makes sense: check the label distribution
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Create the target from the observed outcome
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

print(f"Dataset: {len(df):,} rows × {len(df.columns)} columns")
print(f"Clients: {df['client_id'].nunique()}")
print()
print("Target distribution (is_declining):")
print(df["is_declining"].value_counts().rename({1: "declining (1)", 0: "not declining (0)"}))
print(f"\nBase rate: {df['is_declining'].mean()*100:.1f}% declining")
print("→ Binary classification is appropriate: clear two-class outcome, near-balanced split.")

Dataset: 30,000 rows × 45 columns
Clients: 32

Target distribution (is_declining):
is_declining
declining (1)        16262
not declining (0)    13738
Name: count, dtype: int64

Base rate: 54.2% declining
→ Binary classification is appropriate: clear two-class outcome, near-balanced split.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining` — whether a page's impressions declined > 20% from the previous 30-day window to the most recent 30-day window.**

This label is derived from `trend_direction == "down"`, which itself comes from comparing `impressions_last_30d` vs `impressions_prev_30d`. A page is labeled declining when `(impressions_last_30d - impressions_prev_30d) / impressions_prev_30d < -0.20`.

**Is this observed or defined?** It's a hybrid — and I need to be honest about that:
- The *impressions themselves* are observed (measured by Google Search Console).
- The *threshold* (-20%) is a defined rule that converts a continuous signal into a binary label.

This means the label is **observed at the measurement level** (real impression counts) but **defined at the threshold level** (the -20% cutoff is a business choice, not a natural boundary). That's acceptable for a first model — the alternative would be predicting the raw `trend_pct` as a regression target, which is noisier and harder to act on. But I should remember: the model is learning the threshold-defined pattern, not some deeper truth.

**Leakage guard:** `trend_direction` and `trend_pct` are NEVER features — they *are* the label. The 30-day comparison columns (`impressions_last_30d`, `impressions_prev_30d`, etc.) are also off-limits as features because they directly determine the label.

In [2]:
# Show what the target looks like and where it comes from
print("How the label is built:")
print("  trend_direction values → is_declining mapping:")
print()
for td in ["down", "stable", "up", "new", "flat"]:
    mask = df["trend_direction"] == td
    label_val = 1 if td == "down" else 0
    print(f"  trend_direction = '{td:6s}' → is_declining = {label_val}  ({mask.sum():>6,} rows)")

print()
print("Leakage columns (NEVER features):")
print("  - trend_direction (IS the label)")
print("  - trend_pct (continuous version of the label)")
print("  - impressions_last_30d, impressions_prev_30d (label inputs)")
print("  - clicks_last_30d, clicks_prev_30d, sessions_last_30d, sessions_prev_30d (same window)")

How the label is built:
  trend_direction values → is_declining mapping:

  trend_direction = 'down  ' → is_declining = 1  (16,262 rows)
  trend_direction = 'stable' → is_declining = 0  ( 5,962 rows)
  trend_direction = 'up    ' → is_declining = 0  ( 4,388 rows)
  trend_direction = 'new   ' → is_declining = 0  ( 2,236 rows)
  trend_direction = 'flat  ' → is_declining = 0  ( 1,152 rows)

Leakage columns (NEVER features):
  - trend_direction (IS the label)
  - trend_pct (continuous version of the label)
  - impressions_last_30d, impressions_prev_30d (label inputs)
  - clicks_last_30d, clicks_prev_30d, sessions_last_30d, sessions_prev_30d (same window)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: ROC-AUC** — the area under the receiver operating characteristic curve.

Why ROC-AUC?
1. **Threshold-free:** It evaluates the model's ability to *rank* declining pages above non-declining ones across all possible thresholds, which matches the real use case (editors work down a ranked list).
2. **Interpretable baseline:** A random model scores 0.50. The majority-class classifier (predict everything as declining) also scores 0.50 on AUC. Any honest signal should push above 0.50.
3. **Handles the near-balanced case well:** With a 54/46 split, AUC isn't distorted by class imbalance the way accuracy can be.

**Secondary metric: Precision@K** — of the top-K pages the model flags, how many are actually declining? This directly maps to editor productivity: if an editor refreshes the top 100 flagged pages, what fraction actually needed it?

**What number means "good"?**
- AUC > 0.50 = better than random (minimum bar)
- AUC > 0.60 = the model has learned real signal beyond a naive rule
- AUC > 0.70 = practically useful for prioritization
- Precision@100 > 54.2% (the base rate) = the model's top picks are richer in declining pages than picking at random

In [3]:
# Demonstrate the baseline that the model must beat
base_rate = df["is_declining"].mean()

# Majority-class baseline: predict everything as declining
majority_accuracy = max(base_rate, 1 - base_rate)

print("Success metric: ROC-AUC")
print("=" * 50)
print(f"Base rate (declining):             {base_rate*100:.1f}%")
print(f"Majority-class accuracy baseline:  {majority_accuracy*100:.1f}% (just predict 'declining' for all)")
print(f"AUC of any naive baseline:         0.500 (constant or random predictor)")
print()
print("Thresholds for 'good':")
print("  AUC > 0.50 = better than random")
print("  AUC > 0.60 = real signal learned")
print("  AUC > 0.70 = practically useful for prioritization")
print()
print(f"Secondary: Precision@100 must beat {base_rate*100:.1f}% (the base rate).")
print()
print("Note: A model that just predicts the base rate for everyone gets 54.2% accuracy")
print("but AUC = 0.50 — it cannot distinguish declining from non-declining pages.")
print("Our model needs to beat that by actually ranking pages correctly.")

Success metric: ROC-AUC
Base rate (declining):             54.2%
Majority-class accuracy baseline:  54.2% (just predict 'declining' for all)
AUC of any naive baseline:         0.500 (constant or random predictor)

Thresholds for 'good':
  AUC > 0.50 = better than random
  AUC > 0.60 = real signal learned
  AUC > 0.70 = practically useful for prioritization

Secondary: Precision@100 must beat 54.2% (the base rate).

Note: A model that just predicts the base rate for everyone gets 54.2% accuracy
but AUC = 0.50 — it cannot distinguish declining from non-declining pages.
Our model needs to beat that by actually ranking pages correctly.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page (identified by `content_id`), measured over a trailing 90-day window.**

Each row represents a single pseudonymized page with its keyword context, content properties, 90-day performance metrics, and the observed decline label. The grain is confirmed: `content_id` is unique per row (30,000 unique values in 30,000 rows).

For classification, I keep only columns that are legitimate features (no IDs as features, no label-source columns). Below I show the dataframe with candidate features and the target, plus a sketch of what the target column looks like.

In [4]:
# Confirm the grain: one row = one content page
print("Grain check:")
print(f"  Total rows:          {len(df):,}")
print(f"  Unique content_ids:  {df['content_id'].nunique():,}")
print(f"  → {'✓ One row per content page' if len(df) == df['content_id'].nunique() else '✗ DUPLICATES — investigate!'}")
print()

Grain check:
  Total rows:          30,000
  Unique content_ids:  30,000
  → ✓ One row per content page



In [5]:
# Define the feature groups (no leakage columns, no IDs as features)
# IDs are kept for grouping/splitting only
id_cols = ["content_id", "client_id"]

# Candidate features (safe to use)
keyword_features = ["search_volume", "competition", "cpc"]
content_features = ["word_count", "char_count", "content_age_days", "days_since_last_update"]
performance_features = [
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions"
]
rate_features = ["ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"]
categorical_features = ["content_type", "main_intent", "competition_level"]

# NEVER features (leakage or identifiers-as-features)
leakage_cols = [
    "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d"
]

target_col = "is_declining"

# Build the analysis dataframe
feature_cols = keyword_features + content_features + performance_features + rate_features + categorical_features
analysis_df = df[id_cols + feature_cols + [target_col]].copy()

print(f"Analysis dataframe: {analysis_df.shape[0]:,} rows × {analysis_df.shape[1]} columns")
print(f"  IDs (grouping only):     {len(id_cols)}")
print(f"  Candidate features:      {len(feature_cols)}")
print(f"  Target:                  1 ({target_col})")
print()
analysis_df.head(10)

Analysis dataframe: 30,000 rows × 28 columns
  IDs (grouping only):     2
  Candidate features:      25
  Target:                  1 (is_declining)



,content_id,client_id,search_volume,competition,cpc,word_count,char_count,content_age_days,days_since_last_update,impressions_90d,...,days_with_sessions,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,content_type,main_intent,competition_level,is_declining
0,content_304f48230142,client_f369cb89fc,10.0,0.67,2.05,3221.0,20457.0,187,20,3803,...,13,0.76,10.6,5.88,4.55,0.0,keyword article,transactional,HIGH,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,0.05,2481.0,15562.0,445,25,15320,...,9,0.05,20.3,0.00,10.00,0.0,keyword article,informational,LOW,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,0.00,3515.0,23643.0,141,20,12581,...,11,0.09,36.5,0.00,28.57,0.0,keyword article,informational,LOW,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,0.00,NaN,NaN,463,22,11751,...,51,0.49,6.2,1.28,3.45,0.0,keyword article,commercial,LOW,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,0.00,2803.0,17469.0,263,14,19140,...,33,0.13,44.0,0.00,24.29,0.0,keyword article,informational,LOW,1
5,content_d4084a4bc775,client_f369cb89fc,720.0,1.00,1.05,3080.0,18178.0,147,20,3970,...,5,0.03,8.5,0.00,25.00,0.0,keyword article,transactional,HIGH,1
6,content_9a34b442b552,client_8722616204,0.0,0.00,0.00,3059.0,20810.0,90,20,20,...,1,0.00,7.0,0.00,0.00,0.0,keyword article,informational,LOW,1
7,content_a63219c6e95a,client_19581e27de,590.0,0.44,0.64,NaN,NaN,445,22,1724,...,11,0.06,21.2,3.57,7.14,0.0,keyword article,commercial,MEDIUM,0
8,content_5e6c160719bc,client_6208ef0f77,0.0,0.00,0.00,3807.0,24228.0,90,20,32574,...,44,0.09,46.0,5.88,6.25,0.0,keyword article,informational,LOW,1
9,content_c27558df2b0c,client_19581e27de,0.0,0.00,0.00,NaN,NaN,257,104,1240,...,3,0.16,4.9,0.00,0.00,0.0,keyword article,informational,LOW,1


In [6]:
# Sketch what the target column looks like
print("Target column sketch (is_declining):")
print("=" * 50)
target_summary = pd.DataFrame({
    "is_declining": [0, 1],
    "meaning": ["page is NOT declining (stable/up/new/flat)", "page IS declining (impressions dropped > 20%)"],
    "count": [
        (df["is_declining"] == 0).sum(),
        (df["is_declining"] == 1).sum()
    ],
    "pct": [
        f"{(df['is_declining'] == 0).mean()*100:.1f}%",
        f"{(df['is_declining'] == 1).mean()*100:.1f}%"
    ]
})
print(target_summary.to_string(index=False))
print()
print("Sample rows showing features → target:")
print(df[["content_id", "impressions_90d", "ctr", "avg_position",
          "content_age_days", "days_since_last_update", "content_type",
          "is_declining"]].sample(8, random_state=42).to_string(index=False))

Target column sketch (is_declining):
 is_declining                                       meaning  count   pct
            0    page is NOT declining (stable/up/new/flat)  13738 45.8%
            1 page IS declining (impressions dropped > 20%)  16262 54.2%

Sample rows showing features → target:
          content_id  impressions_90d   ctr  avg_position  content_age_days  days_since_last_update       content_type  is_declining
content_9824710082d8              283  0.00          21.3               174                     104    keyword article             0
content_3efa3a7c46bb             8878  0.09           8.8               134                      20    keyword article             0
content_575dc8a2ab0f                3  0.00           0.0               109                       8     feedly article             0
content_0dbd6911ba04              124  0.00           8.1               151                      20 comparison article             1
content_bbaf87019afb             4294  

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The current rule is simple: *"flag a page as declining if its impressions dropped more than 20% from the previous 30 days."* That rule already exists as `trend_direction == "down"`. So why would we need ML at all?

**Three reasons ML beats this fixed rule:**

1. **The rule is backward-looking; ML can be forward-looking.** The -20% rule only fires *after* the decline has already happened. A model trained on 90-day features (position drift, engagement patterns, content staleness) could flag pages that are *likely to decline next month* — catching problems earlier.

2. **Many signals interact, and the interactions differ by content type.** A keyword article with high search volume, poor position, and stale content declines for different reasons than a feedly article with low engagement and no keyword data. A single threshold can't capture these multi-signal interactions. ML can — that's literally what tree-based models do.

3. **A rule gives a binary flag; a model gives a ranked queue.** The rule says "declining: yes/no." It flags 16,262 pages equally. But an editor can't refresh 16,000 pages at once — they need to know *which declining pages to fix first*. A model's predicted probabilities produce a natural ranking from "almost certainly declining" to "probably fine," which is directly usable as a work queue.

The code below demonstrates reason #2 concretely: decline rates vary across content types and position tiers in non-additive ways that a single rule can't capture.

In [7]:
# Demonstrate why a single rule can't capture the pattern:
# decline rates vary by content_type x position_tier interaction

print("Decline rate by content_type x position_tier (interactions a rule can't capture):")
print("=" * 80)

cross = pd.crosstab(
    df["content_type"],
    df["position_tier"],
    values=df["is_declining"],
    aggfunc="mean"
).round(3) * 100

# Reorder columns logically
tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep", "no_data"]
cross = cross[[c for c in tier_order if c in cross.columns]]

print(cross.to_string())
print()
print("→ The decline rate isn't a simple function of position OR content type alone.")
print("  e.g., 'keyword article' at 'striking' position has a different decline pattern")
print("  than 'comparison article' at 'striking'. A single threshold misses this.")
print()

# Also show: the number of features that correlate with decline
numeric_features = keyword_features + content_features + performance_features + rate_features
correlations = df[numeric_features + ["is_declining"]].corr()["is_declining"].drop("is_declining")
significant = correlations[correlations.abs() > 0.05].sort_values(key=abs, ascending=False)

print(f"Features with |correlation| > 0.05 to is_declining: {len(significant)} of {len(numeric_features)}")
print(significant.round(3).to_string())
print()
print("→ Multiple signals carry information. A model can combine them;")
print("  a fixed rule would need to be hand-tuned for each combination.")

Decline rate by content_type x position_tier (interactions a rule can't capture):
position_tier       top_3  page_1  striking  page_3_5  deep
content_type                                               
comparison article  100.0    57.0      50.6      72.2   NaN
feedly article        4.0    47.0      51.9      48.5  60.0
keyword article      37.2    57.8      61.5      56.1  34.1

→ The decline rate isn't a simple function of position OR content type alone.
  e.g., 'keyword article' at 'striking' position has a different decline pattern
  than 'comparison article' at 'striking'. A single threshold misses this.

Features with |correlation| > 0.05 to is_declining: 6 of 22
days_with_impressions     0.190
content_age_days         -0.164
word_count                0.090
days_since_last_update    0.081
char_count                0.072
ctr                      -0.062

→ Multiple signals carry information. A model can combine them;
  a fixed rule would need to be hand-tuned for each combination.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.